# 7. Ablation Synthetic Data

Σε αυτό το notebook εξετάζεται η επίδραση των synthetic δεδομένων στην απόδοση του hallucination detection συστήματος.

Συγκρίνονται δύο βασικά σενάρια:

1. **Trial only**: εκπαίδευση μόνο στα 80 human-labeled trial samples.
2. **Trial + Synthetic**: εκπαίδευση στα 80 human-labeled samples μαζί με 518 synthetic samples.

Η αξιολόγηση γίνεται στο ίδιο labeled evaluation set των 1500 samples, χρησιμοποιώντας τα ίδια NLI και semantic similarity features. Έτσι, η σύγκριση εστιάζει αποκλειστικά στην επίδραση των synthetic δεδομένων.

## 1. Load datasets

Σε αυτό το βήμα φορτώνονται τα δύο βασικά αρχεία του ablation:

- `4.nli_similarity_features_dataset.csv`: original labeled evaluation set με 1500 samples.
- `6.nli_similarity_features_dataset.csv`: augmented training set με 598 samples.

Το πρώτο χρησιμοποιείται μόνο για αξιολόγηση, ενώ το δεύτερο χρησιμοποιείται για την εκπαίδευση των μοντέλων.

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr

# Paths
# Αν το notebook βρίσκεται μέσα στον ίδιο φάκελο, τότε το Path.cwd() αρκεί.
# Αν το τρέξεις από αλλού, χρησιμοποίησε το Windows path.
DATA_DIR = Path.cwd()

if not (DATA_DIR / "4.nli_similarity_features_dataset.csv").exists():
    DATA_DIR = Path((__import__('os').environ.get('SHROOM_PROJECT_ROOT', '.') + '/data/7. Ablation Synthetic Data'))

EVAL_PATH = DATA_DIR / "4.nli_similarity_features_dataset.csv"
AUG_PATH = DATA_DIR / "6.nli_similarity_features_dataset.csv"

# Load datasets
eval_df = pd.read_csv(EVAL_PATH)
aug_df = pd.read_csv(AUG_PATH)

print("Evaluation dataset shape:", eval_df.shape)
print("Augmented dataset shape:", aug_df.shape)

print("\nEvaluation label distribution:")
print(eval_df["label"].value_counts())

print("\nAugmented source_type distribution:")
print(aug_df["source_type"].value_counts())

print("\nAugmented label distribution:")
print(aug_df["label"].value_counts())

## 2. Feature columns

Χρησιμοποιούνται τα ίδια feature columns και στα δύο πειράματα, ώστε η σύγκριση να είναι δίκαιη.

Τα features περιλαμβάνουν:

- NLI scores μεταξύ `src`, `tgt` και `hyp`.
- Semantic similarity scores μεταξύ `src`, `tgt` και `hyp`.

Ο στόχος είναι να εξεταστεί αποκλειστικά η επίδραση των synthetic δεδομένων και όχι η επίδραση διαφορετικών features.

In [ ]:
# Feature columns
feature_cols = [
    "src_ent", "src_neu", "src_con",
    "tgt_ent", "tgt_neu", "tgt_con",
    "sim_src_hyp", "sim_tgt_hyp", "sim_src_tgt",
    "sim_max", "sim_min", "sim_mean",
    "sim_gap_tgt_src", "sim_hyp_distance_from_truth"
]

# Safety checks
for col in feature_cols:
    assert col in eval_df.columns, f"Missing column in eval_df: {col}"
    assert col in aug_df.columns, f"Missing column in aug_df: {col}"

required_eval_cols = ["p_hallucination", "label_bin", "label", "task"]
required_aug_cols = ["p_hallucination", "label_bin", "label", "task", "source_type"]

for col in required_eval_cols:
    assert col in eval_df.columns, f"Missing column in eval_df: {col}"

for col in required_aug_cols:
    assert col in aug_df.columns, f"Missing column in aug_df: {col}"

print("All required columns found.")

## 3. Trial only vs Trial + Synthetic

Εδώ δημιουργούνται τα δύο training sets του ablation.

Στο πρώτο πείραμα χρησιμοποιούνται μόνο τα human-labeled trial samples.

Στο δεύτερο πείραμα χρησιμοποιείται ολόκληρο το augmented training set, δηλαδή τα human-labeled samples μαζί με τα synthetic samples.

Και τα δύο μοντέλα αξιολογούνται στο ίδιο evaluation set των 1500 samples.

In [ ]:
# Prepare train/test splits

# Experiment A: μόνο τα 80 human-labeled trial samples
trial_train_df = aug_df[aug_df["source_type"] == "human_labeled"].copy()

# Experiment B: όλα τα 598 samples = 80 human + 518 synthetic
augmented_train_df = aug_df.copy()

print("Trial-only training shape:", trial_train_df.shape)
print("Trial + Synthetic training shape:", augmented_train_df.shape)

print("\nTrial-only task distribution:")
print(trial_train_df["task"].value_counts())

print("\nTrial + Synthetic task distribution:")
print(augmented_train_df["task"].value_counts())

## 4. Training and evaluation function

Η συνάρτηση `train_and_evaluate` εκπαιδεύει έναν Random Forest Regressor και αξιολογεί τις προβλέψεις του.

Το μοντέλο προβλέπει πρώτα την πιθανότητα `p(Hallucination)` και στη συνέχεια η πιθανότητα μετατρέπεται σε binary label με threshold 0.5.

Οι βασικές μετρικές αξιολόγησης είναι:

- Accuracy
- Spearman correlation
- Mean Absolute Error

Επιπλέον, υπολογίζονται αποτελέσματα ανά task και feature importance.

In [ ]:
# Evaluation function
#Παιρνει ως ορισματα:
# - train_df: το DataFrame που θα χρησιμοποιηθεί για εκπαίδευση
# - test_df: το DataFrame που θα χρησιμοποιηθεί για αξιολόγηση
# - experiment_name: ένα string που περιγράφει το πείραμα (π.χ. "Trial Only", "Trial + Synthetic")
# - sample_weight: (προεραιτικο) ένας πίνακας με βάρη για τα δείγματα εκπαίδευσης, που μπορεί να χρησιμοποιηθεί για να δώσει μεγαλύτερη βαρύτητα στα human-labeled δείγματα
def train_and_evaluate(train_df, test_df, experiment_name, sample_weight=None):
    #Ετοιμάζει τα training features και target
    X_train = train_df[feature_cols].astype(float).values
    y_train = train_df["p_hallucination"].astype(float).values

    #Ετοιμάζει το test set
    X_test = test_df[feature_cols].astype(float).values
    y_test_reg = test_df["p_hallucination"].astype(float).values
    y_test_bin = test_df["label_bin"].astype(int).values

    #Φτιάχνει το Random Forest Regressor
    model = RandomForestRegressor(
        #500 δέντρα
        n_estimators=500,
        #κάνει το αποτέλεσμα αναπαράξιμο
        random_state=42,
        #χρησιμοποιεί όλους τους διαθέσιμους CPU cores
        n_jobs=-1
    )

    #Εκπαιδεύει το μοντέλο
    model.fit(X_train, y_train, sample_weight=sample_weight)

    #Παράγει προβλέψεις πιθανότητας
    pred_reg = model.predict(X_test)
    pred_reg = np.clip(pred_reg, 0.0, 1.0)

    #Μετατρέπει πιθανότητα σε binary label
    pred_bin = (pred_reg >= 0.5).astype(int)

    #Υπολογίζει τα βασικά metrics
    acc = accuracy_score(y_test_bin, pred_bin)
    sp = spearmanr(y_test_reg, pred_reg).correlation
    mae = mean_absolute_error(y_test_reg, pred_reg)

    #Φτιάχνει το summary dictionary με τα συνολικά αποτελέσματα
    summary = {
        "experiment": experiment_name,
        "n_train": len(train_df),
        "accuracy": acc,
        "spearman": sp,
        "mae": mae
    }

    #Κανει per-task evaluation
    per_task_rows = []

    temp = test_df.copy()
    temp["pred_p_hallucination"] = pred_reg
    temp["pred_label_bin"] = pred_bin

    for task, task_df in temp.groupby("task"):
        task_y_reg = task_df["p_hallucination"].astype(float).values
        task_y_bin = task_df["label_bin"].astype(int).values
        task_pred_reg = task_df["pred_p_hallucination"].values
        task_pred_bin = task_df["pred_label_bin"].astype(int).values

        task_acc = accuracy_score(task_y_bin, task_pred_bin)
        task_sp = spearmanr(task_y_reg, task_pred_reg).correlation
        task_mae = mean_absolute_error(task_y_reg, task_pred_reg)

        per_task_rows.append({
            "experiment": experiment_name,
            "task": task,
            "n_test": len(task_df),
            "accuracy": task_acc,
            "spearman": task_sp,
            "mae": task_mae
        })

    prediction_cols = [
        "id", "task", "src", "tgt", "hyp",
        "label", "label_bin", "p_hallucination",
        "pred_p_hallucination", "pred_label_bin"
    ]
    prediction_cols = [col for col in prediction_cols if col in temp.columns]

    #Φτιάχνει predictions dataframe για να το χρησιμοποιήσουμε αργότερα για error analysis
    predictions_df = temp[prediction_cols].copy()
    predictions_df["experiment"] = experiment_name
    predictions_df["correct"] = predictions_df["label_bin"] == predictions_df["pred_label_bin"]

    #Υπολογίζει feature importance
    importance_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)

    importance_df["experiment"] = experiment_name

    return summary, pd.DataFrame(per_task_rows), predictions_df, importance_df, model

## 5. Main ablation experiment

Σε αυτό το σημείο τρέχουν τα δύο βασικά πειράματα:

1. **Trial only**
2. **Trial + Synthetic**

Η σύγκριση δείχνει αν η προσθήκη synthetic δεδομένων βελτιώνει ή χειροτερεύει την απόδοση του μοντέλου.

In [ ]:
# Run experiments
trial_summary, trial_per_task, trial_preds, trial_importance, trial_model = train_and_evaluate(
    train_df=trial_train_df,
    test_df=eval_df,
    experiment_name="Trial only"
)

aug_summary, aug_per_task, aug_preds, aug_importance, aug_model = train_and_evaluate(
    train_df=augmented_train_df,
    test_df=eval_df,
    experiment_name="Trial + Synthetic"
)

## 6. Overall results

Ο πίνακας `summary_df` παρουσιάζει τη συνολική απόδοση των δύο πειραμάτων.

Αυτός είναι ο βασικός πίνακας του ablation, γιατί δείχνει αν το synthetic data βοήθησε συνολικά ή όχι.

In [ ]:
# Overall results
summary_df = pd.DataFrame([trial_summary, aug_summary])

print("=== Overall Ablation Results ===")
display(summary_df)

summary_df.to_csv(DATA_DIR / "ablation_summary_results.csv", index=False)

## 7. Per-task results

Ο πίνακας `per_task_df` παρουσιάζει την απόδοση ξεχωριστά για κάθε task:

- DM
- MT
- PG

Αυτή η ανάλυση δείχνει αν το synthetic data επηρεάζει όλα τα tasks με τον ίδιο τρόπο ή αν βοηθάει/βλάπτει συγκεκριμένα tasks περισσότερο.

In [ ]:
# Per-task results
per_task_df = pd.concat([trial_per_task, aug_per_task], ignore_index=True)

print("=== Per-task Results ===")
display(per_task_df)

per_task_df.to_csv(DATA_DIR / "ablation_per_task_results.csv", index=False)

## 8. Feature importance

Η ανάλυση feature importance δείχνει σε ποια χαρακτηριστικά βασίζεται περισσότερο το Random Forest.

Η σύγκριση ανάμεσα στο Trial-only και στο Trial + Synthetic μοντέλο βοηθάει να καταλάβουμε αν η προσθήκη synthetic δεδομένων αλλάζει τη συμπεριφορά του μοντέλου.

In [ ]:
# Feature importance
importance_df = pd.concat([trial_importance, aug_importance], ignore_index=True)

print("=== Feature Importance ===")
display(importance_df)

importance_df.to_csv(DATA_DIR / "ablation_feature_importance.csv", index=False)

## 9. Save predictions

Οι προβλέψεις των δύο μοντέλων αποθηκεύονται ώστε να χρησιμοποιηθούν στο error analysis.

Το αρχείο `ablation_predictions.csv` περιέχει για κάθε sample την πραγματική ετικέτα, την προβλεπόμενη πιθανότητα hallucination, το binary prediction και αν η πρόβλεψη ήταν σωστή.

In [ ]:
# Save predictions
predictions_df = pd.concat([trial_preds, aug_preds], ignore_index=True)
predictions_df.to_csv(DATA_DIR / "ablation_predictions.csv", index=False)

print("Saved files:")
print(DATA_DIR / "ablation_summary_results.csv")
print(DATA_DIR / "ablation_per_task_results.csv")
print(DATA_DIR / "ablation_feature_importance.csv")
print(DATA_DIR / "ablation_predictions.csv")

## 10. Error analysis

Στο error analysis συγκρίνονται τα λάθη των δύο μοντέλων.

Συγκεκριμένα εξετάζονται:

- πόσα λάθη κάνει το Trial-only μοντέλο,
- πόσα λάθη κάνει το Trial + Synthetic μοντέλο,
- πόσα λάθη διορθώνονται με την προσθήκη synthetic data,
- πόσα νέα λάθη εισάγονται από το synthetic data.

Αυτή η ανάλυση βοηθάει να εξηγηθεί γιατί το synthetic data βελτίωσε ή χειροτέρεψε την απόδοση.

In [ ]:
# Error analysis
trial_errors = predictions_df[
    (predictions_df["experiment"] == "Trial only") &
    (predictions_df["correct"] == False)
].copy()

aug_errors = predictions_df[
    (predictions_df["experiment"] == "Trial + Synthetic") &
    (predictions_df["correct"] == False)
].copy()

print("Trial-only errors:", len(trial_errors))
print("Trial + Synthetic errors:", len(aug_errors))

# Samples wrong in Trial only
trial_wrong_ids = set(trial_errors["id"])

# Samples wrong in Trial + Synthetic
aug_wrong_ids = set(aug_errors["id"])

# Errors fixed by synthetic data
fixed_by_synthetic = trial_errors[
    ~trial_errors["id"].isin(aug_wrong_ids)
].copy()

# New errors introduced by synthetic data
introduced_by_synthetic = aug_errors[
    ~aug_errors["id"].isin(trial_wrong_ids)
].copy()

print("\nFixed by synthetic data:", len(fixed_by_synthetic))
print("Introduced by synthetic data:", len(introduced_by_synthetic))

fixed_by_synthetic.to_csv(DATA_DIR / "errors_fixed_by_synthetic.csv", index=False)
introduced_by_synthetic.to_csv(DATA_DIR / "errors_introduced_by_synthetic.csv", index=False)

## 11. Fixed and introduced errors

Σε αυτό το βήμα μετράμε αναλυτικά τα λάθη που διορθώθηκαν και τα νέα λάθη που εισήχθησαν από το synthetic data.

Η ανάλυση γίνεται ανά task και ανά πραγματικό label, ώστε να δούμε αν το synthetic data δημιουργεί συγκεκριμένο είδος σφάλματος.

In [ ]:
# Quantify fixed / introduced errors
print("Fixed by synthetic - by task:")
display(fixed_by_synthetic["task"].value_counts())

print("Introduced by synthetic - by task:")
display(introduced_by_synthetic["task"].value_counts())

print("Fixed by synthetic - true labels:")
display(fixed_by_synthetic["label"].value_counts())

print("Introduced by synthetic - true labels:")
display(introduced_by_synthetic["label"].value_counts())

print("Introduced by synthetic - predicted labels:")
display(introduced_by_synthetic["pred_label_bin"].value_counts())

## 12. False positive / false negative analysis

Εδώ διαχωρίζονται τα λάθη σε:

- **False Positives**: σωστά samples που προβλέπονται λανθασμένα ως hallucinations.
- **False Negatives**: hallucinated samples που προβλέπονται λανθασμένα ως not hallucinations.

Αυτό είναι σημαντικό γιατί δείχνει αν το synthetic data κάνει το μοντέλο υπερβολικά αυστηρό ή υπερβολικά χαλαρό.

In [ ]:
# ============================================================
# False positive / false negative analysis
# ============================================================

introduced_by_synthetic["error_type"] = np.where(
    (introduced_by_synthetic["label_bin"] == 0) & (introduced_by_synthetic["pred_label_bin"] == 1),
    "False Positive",
    "False Negative"
)

fixed_by_synthetic["error_type"] = np.where(
    (fixed_by_synthetic["label_bin"] == 0) & (fixed_by_synthetic["pred_label_bin"] == 1),
    "False Positive",
    "False Negative"
)

print("Fixed errors by type:")
display(fixed_by_synthetic["error_type"].value_counts())

print("Introduced errors by type:")
display(introduced_by_synthetic["error_type"].value_counts())

print("Introduced errors by task and type:")
introduced_error_table = pd.crosstab(introduced_by_synthetic["task"], introduced_by_synthetic["error_type"])
display(introduced_error_table)

print("Fixed errors by task and type:")
fixed_error_table = pd.crosstab(fixed_by_synthetic["task"], fixed_by_synthetic["error_type"])
display(fixed_error_table)

introduced_error_table.to_csv(DATA_DIR / "introduced_errors_by_task_and_type.csv")
fixed_error_table.to_csv(DATA_DIR / "fixed_errors_by_task_and_type.csv")

## 13. Error analysis conclusion

Με βάση τα αποτελέσματα που προέκυψαν από το τρέχον run:

- Το Trial-only μοντέλο έκανε **386 λάθη**.
- Το Trial + Synthetic μοντέλο έκανε **433 λάθη**.
- Το synthetic data διόρθωσε **77 λάθη**.
- Το synthetic data εισήγαγε **124 νέα λάθη**.

Η προσθήκη synthetic δεδομένων δεν οδήγησε σε συνολική βελτίωση. Παρότι διόρθωσε ορισμένα λάθη του Trial-only μοντέλου, εισήγαγε περισσότερα νέα λάθη από όσα διόρθωσε.

Η πλειονότητα των νέων λαθών ήταν **False Positives**. Αυτό δείχνει ότι το μοντέλο με synthetic data έγινε πιο ευαίσθητο στην ανίχνευση hallucination και τείνει να χαρακτηρίζει σωστές παραφράσεις ως hallucinations.

Το φαινόμενο ήταν ιδιαίτερα έντονο στο MT task, όπου σχεδόν όλα τα νέα λάθη ήταν False Positives.

## 14. Controlled synthetic ratio ablation

Σε αυτό το πείραμα εξετάζεται αν η χρήση μικρότερης ποσότητας synthetic data μπορεί να βοηθήσει.

Αντί να χρησιμοποιηθούν όλα τα synthetic samples, δοκιμάζονται διαφορετικά ποσοστά:

- 0%
- 25%
- 50%
- 75%
- 100%

Ο στόχος είναι να φανεί αν το synthetic data είναι χρήσιμο μόνο όταν χρησιμοποιείται σε περιορισμένη ποσότητα.

In [ ]:
# ============================================================
# Controlled synthetic ratio ablation
# ============================================================

human_df = aug_df[aug_df["source_type"] == "human_labeled"].copy()
synthetic_df = aug_df[aug_df["source_type"] != "human_labeled"].copy()

print("Human samples:", len(human_df))
print("Synthetic samples:", len(synthetic_df))

synthetic_ratios = [0.0, 0.25, 0.5, 0.75, 1.0]
ratio_results = []

for ratio in synthetic_ratios:
    if ratio == 0.0:
        train_df = human_df.copy()
    elif ratio == 1.0:
        train_df = pd.concat([human_df, synthetic_df], ignore_index=True)
    else:
        n_synth = int(round(len(synthetic_df) * ratio))

        _, sampled_synth = train_test_split(
            synthetic_df,
            test_size=n_synth,
            stratify=synthetic_df["label"],
            random_state=42
        )

        train_df = pd.concat([human_df, sampled_synth], ignore_index=True)

    summary, _, _, _, _ = train_and_evaluate(
        train_df=train_df,
        test_df=eval_df,
        experiment_name=f"Trial + {int(ratio * 100)}% Synthetic"
    )

    summary["synthetic_ratio"] = ratio
    summary["n_human"] = len(human_df)
    summary["n_synthetic"] = len(train_df) - len(human_df)
    ratio_results.append(summary)

ratio_results_df = pd.DataFrame(ratio_results)

# Reorder columns for readability
ratio_results_df = ratio_results_df[
    ["experiment", "synthetic_ratio", "n_train", "n_human", "n_synthetic", "accuracy", "spearman", "mae"]
]

display(ratio_results_df)

ratio_results_df.to_csv(DATA_DIR / "ablation_synthetic_ratio_results.csv", index=False)

## 15. Weighted synthetic data ablation

Σε αυτό το πείραμα χρησιμοποιούνται όλα τα synthetic samples, αλλά με μικρότερο βάρος από τα human-labeled samples.

Τα human-labeled samples έχουν βάρος 1.0, ενώ τα synthetic samples δοκιμάζονται με διαφορετικά βάρη.

Ο στόχος είναι να εξεταστεί αν το synthetic data μπορεί να προσφέρει χρήσιμο σήμα χωρίς να κυριαρχεί στην εκπαίδευση.

In [ ]:
# ============================================================
# Weighted synthetic data ablation
# ============================================================

synthetic_weights = [0.1, 0.25, 0.5, 0.75, 1.0]
weighted_results = []

train_df = aug_df.copy()

for synth_weight in synthetic_weights:
    sample_weights = np.where(
        train_df["source_type"] == "human_labeled",
        1.0,
        synth_weight
    )

    summary, _, _, _, _ = train_and_evaluate(
        train_df=train_df,
        test_df=eval_df,
        experiment_name=f"Weighted Synthetic {synth_weight}x",
        sample_weight=sample_weights
    )

    summary["human_weight"] = 1.0
    summary["synthetic_weight"] = synth_weight
    weighted_results.append(summary)

weighted_results_df = pd.DataFrame(weighted_results)

weighted_results_df = weighted_results_df[
    ["experiment", "n_train", "human_weight", "synthetic_weight", "accuracy", "spearman", "mae"]
]

display(weighted_results_df)

weighted_results_df.to_csv(DATA_DIR / "ablation_synthetic_weight_results.csv", index=False)

## 16. Final comparison and conclusions

Σε αυτό το τελικό βήμα συγκρίνονται όλα τα αποτελέσματα του synthetic data ablation.

Η σύγκριση περιλαμβάνει:

- Trial only
- Trial + Synthetic
- διαφορετικά ποσοστά synthetic data
- διαφορετικά βάρη synthetic data

Με βάση αυτά τα αποτελέσματα αποφασίζεται αν το synthetic data θα χρησιμοποιηθεί στην τελική μεθοδολογία ή αν θα παρουσιαστεί ως αρνητικό αλλά χρήσιμο ablation.

In [ ]:
# ============================================================
# Final comparison helper
# ============================================================

print("Main ablation:")
display(summary_df)

if "ratio_results_df" in globals():
    print("\nSynthetic ratio ablation:")
    display(ratio_results_df)

if "weighted_results_df" in globals():
    print("\nWeighted synthetic data ablation:")
    display(weighted_results_df)

print("\nBest result among ratio experiments:")
if "ratio_results_df" in globals():
    display(ratio_results_df.sort_values(["spearman", "accuracy"], ascending=False).head(1))

print("\nBest result among weighted experiments:")
if "weighted_results_df" in globals():
    display(weighted_results_df.sort_values(["spearman", "accuracy"], ascending=False).head(1))